<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
سپر صوتی انتقام جویان
</font>
</h1>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>

</font>
</p>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
در این قسمت کتاب‌خانه‌ها و ابزار مورد نیاز خود را <code>import</code> کنید و فایل داده‌ها را که در پوشه‌ی <code>Data</code> و که با نام‌های <code>train.csv</code> و <code>test.csv</code> ذخیره‌شده‌اند را بخوانید و وارد محیط کار خود کنید.
</font>
</p>

In [1]:
import os
import random
import time
import zipfile
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.ndimage import label
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupKFold


# Reproducible, CPU-friendly configuration
SEED = 42
SR = 22050
HOP_LENGTH = 512
DT = HOP_LENGTH / SR
ENSEMBLE_WEIGHTS = (0.10, 0.35, 0.55)
MASK_THRESHOLD = 0.35
MIN_EVENT_FRAMES = 15
MERGE_GAP = 5
REFINER_RADIUS = 20
REFINER_FOLDS = 5
REFINER_ENSEMBLE_WEIGHTS = (0.20, 0.40, 0.40)
START_OFFSET_SCALE = 0.99
END_OFFSET_SCALE = 0.98

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(8)
device = torch.device("cpu")


def first_existing(candidates):
    return next((path for path in candidates if os.path.exists(path)), candidates[0])


TRAIN_CSV = first_existing(("train_labels.csv", "train.csv", "Data/train_labels.csv", "Data/train.csv"))
TRAIN_DIR = first_existing(("train", "Data/train", "data/train"))
TEST_DIR = first_existing(("test", "Data/test", "data/test"))



train_frame = pd.read_csv(TRAIN_CSV)
print(f"Training rows: {len(train_frame)}")
print("Drone-count distribution:")
print(train_frame.drone_count.value_counts().sort_index())

Training rows: 2100
Drone-count distribution:
drone_count
0    208
1    718
2    624
3    350
4    200
Name: count, dtype: int64


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
        در این سوال شما می‌توانید از هر تکنیک پیش‌پردازش/مهندسی ویژگی دلخواهتان، استفاده کنید.
    <br>
    تکنیک‌هایی که استفاده می‌کنید به شکل مستقیم مورد ارزیابی توسط سامانه داوری قرار <b>نمی‌گیرند.</b> بلکه همه آن‌ها در دقت مدل شما تاثیر خواهند گذاشت؛ بنابراین هر چه پیش‌پردازش/مهندسی ویژگی بهتری انجام دهید تا دقت مدل بهبود پیدا کند، امتیاز بیشتری از این سوال کسب خواهید کرد.
    در این قسمت شما می‌توانید بخشی از داده‌ی موجود را برای اعتبارسنجی در نظر بگیرید.
</font>
</p>

In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.conv(x) + self.residual(x)


class DroneTCN(nn.Module):
    def __init__(self, kernel_size=5):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(128, 64, 7, padding=3), nn.BatchNorm1d(64), nn.GELU())
        self.b1 = ConvBlock(64, 64, kernel_size, 1)
        self.b2 = ConvBlock(64, 64, kernel_size, 2)
        self.b3 = ConvBlock(64, 64, kernel_size, 4)
        self.b4 = ConvBlock(64, 64, kernel_size, 8)
        self.b5 = ConvBlock(64, 64, kernel_size, 16)
        self.head = nn.Sequential(
            nn.Conv1d(64, 32, 5, padding=2), nn.BatchNorm1d(32), nn.GELU(), nn.Conv1d(32, 1, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        return self.head(x).squeeze(1)


def dice_loss(logits, targets, smooth=1.0):
    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=-1)
    denominator = probabilities.sum(dim=-1) + targets.sum(dim=-1)
    return 1.0 - ((2.0 * intersection + smooth) / (denominator + smooth)).mean()


def segmentation_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss(logits, targets)


def parse_intervals(value):
    if pd.isna(value) or not str(value):
        return []
    return [tuple(map(float, item.split("-"))) for item in str(value).split(";")]


def load_training_data(frame):
    images = []
    targets = []
    intervals = []
    groups = defaultdict(list)
    for row in frame.itertuples(index=False):
        image = cv2.imread(os.path.join(TRAIN_DIR, row.image_file), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(row.image_file)
        width = image.shape[1]
        target = np.zeros(width, dtype=np.float32)
        row_intervals = parse_intervals(row.intervals)
        for start, end in row_intervals:
            first = max(0, min(width - 1, round(start / DT)))
            last = max(0, min(width, round(end / DT)))
            target[first:last] = 1.0
        groups[width].append(len(images))
        images.append(image.astype(np.float32) / 255.0)
        targets.append(target)
        intervals.append(row_intervals)
    return images, targets, intervals, groups


MODEL_CONFIGS = (
    ("drone_model_1.pth", 5, 42, 12, 1.5e-3),
    ("drone_model_2.pth", 7, 43, 12, 1.5e-3),
    ("drone_model_3.pth", 5, 101, 12, 1.2e-3),
)


def prepare_base_models(images, targets, groups):
    models = []
    for filename, kernel_size, seed, epochs, learning_rate in MODEL_CONFIGS:
        # Set the seed before construction so it controls initialization too.
        torch.manual_seed(seed)
        np.random.seed(seed)
        model = DroneTCN(kernel_size).to(device)
        if os.path.exists(filename):
            model.load_state_dict(torch.load(filename, map_location=device, weights_only=True))
            print(f"Loaded {filename}")
        else:
            optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
            for epoch in range(1, epochs + 1):
                started = time.time()
                model.train()
                batches = []
                for indices in groups.values():
                    np.random.shuffle(indices)
                    batches.extend(indices[i:i + 32] for i in range(0, len(indices), 32))
                np.random.shuffle(batches)
                running_loss = 0.0
                for indices in batches:
                    x = torch.from_numpy(np.stack([images[i] for i in indices])).to(device)
                    y = torch.from_numpy(np.stack([targets[i] for i in indices])).to(device)
                    loss = segmentation_loss(model(x), y)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item() * len(indices)
                scheduler.step()
                print(f"{filename} epoch {epoch:02d}/{epochs}: "
                      f"loss={running_loss / len(images):.5f}, seconds={time.time() - started:.1f}")
            torch.save(model.state_dict(), filename)
        model.eval()
        models.append(model)
    return models


def ensemble_probabilities(models, images, groups):
    probabilities = [None] * len(images)
    with torch.inference_mode():
        for indices in groups.values():
            for start in range(0, len(indices), 32):
                batch_indices = indices[start:start + 32]
                batch = torch.from_numpy(np.stack([images[i] for i in batch_indices])).to(device)
                output = sum(weight * torch.sigmoid(model(batch))
                             for weight, model in zip(ENSEMBLE_WEIGHTS, models))
                for index, probability in zip(batch_indices, output.cpu().numpy()):
                    probabilities[index] = probability
    return probabilities


def mask_bounds(probability):
    binary = (probability >= MASK_THRESHOLD).astype(np.uint8)
    binary = cv2.morphologyEx(binary[None, :], cv2.MORPH_CLOSE, np.ones(MERGE_GAP, np.uint8))[0]
    labeled, count = label(binary)
    result = []
    for component in range(1, count + 1):
        indices = np.flatnonzero(labeled == component)
        if len(indices) >= MIN_EVENT_FRAMES:
            result.append((indices[0], indices[-1] + 1))
    return result


def boundary_feature(image, probability, position, boundary_type):
    offsets = np.arange(-REFINER_RADIUS, REFINER_RADIUS + 1)
    columns = np.clip(position + offsets, 0, image.shape[1] - 1)
    raw = image[:, columns].astype(np.float32)
    row_mean = image.mean(axis=1, keepdims=True).astype(np.float32)
    row_std = image.std(axis=1, keepdims=True).astype(np.float32) + 1e-3
    normalized = (raw - row_mean) / row_std
    raw_bands = raw.reshape(16, 8, -1).mean(axis=1)
    normalized_bands = normalized.reshape(16, 8, -1).mean(axis=1)
    local_probability = probability[columns][None]
    orientation = np.full((1, len(columns)), 1 if boundary_type == "start" else -1, np.float32)
    relative_position = np.full((1, len(columns)), position / image.shape[1], np.float32)
    return np.concatenate((raw_bands, normalized_bands, local_probability,
                           orientation, relative_position)).astype(np.float32)


class BoundaryRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(35, 48, 5, padding=2), nn.BatchNorm1d(48), nn.GELU(),
            nn.Conv1d(48, 48, 5, padding=4, dilation=2), nn.BatchNorm1d(48), nn.GELU(),
            nn.Conv1d(48, 32, 5, padding=8, dilation=4), nn.BatchNorm1d(32), nn.GELU(),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * (2 * REFINER_RADIUS + 1), 128),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.regressor(self.features(x)).squeeze(1)


def interval_iou(first, second):
    intersection = max(0.0, min(first[1], second[1]) - max(first[0], second[0]))
    union = max(first[1], second[1]) - min(first[0], second[0])
    return intersection / union if union else 0.0


def build_refiner_samples(image_files, images, probabilities, ground_truth):
    features, targets, image_groups = [], [], []
    for image_index, (image, probability, truth) in enumerate(zip(images, probabilities, ground_truth)):
        bounds = mask_bounds(probability)
        if not truth or len(truth) != len(bounds):
            continue
        predicted = [(start * DT, end * DT) for start, end in bounds]
        matrix = np.array([[interval_iou(gt, pred) for pred in predicted] for gt in truth])
        rows, columns = linear_sum_assignment(-matrix)
        for row, column in zip(rows, columns):
            if matrix[row, column] <= 0:
                continue
            gt_start, gt_end = truth[row]
            start, end = bounds[column]
            candidates = (
                ("start", start, gt_start / DT - start),
                ("end", end, gt_end / DT - end),
            )
            for boundary_type, position, target in candidates:
                if abs(target) <= 5:
                    features.append(boundary_feature(images[image_index], probability, position, boundary_type))
                    targets.append(target)
                    image_groups.append(image_index)
    return np.stack(features), np.asarray(targets, np.float32), np.asarray(image_groups)


def train_refiner_fold(x_train, y_train, x_valid, y_valid, fold, variant,
                       learning_rate, batch_seed, loss_kind, beta):
    model = BoundaryRefiner().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    train_x, train_y = torch.from_numpy(x_train), torch.from_numpy(y_train)
    valid_x, valid_y = torch.from_numpy(x_valid), torch.from_numpy(y_valid)
    generator = torch.Generator().manual_seed(batch_seed + fold)
    best_loss, best_state, patience = float("inf"), None, 0
    for epoch in range(30):
        model.train()
        order = torch.randperm(len(train_x), generator=generator)
        for start in range(0, len(train_x), 256):
            indices = order[start:start + 256]
            prediction = model(train_x[indices])
            if loss_kind == "l1":
                loss = F.l1_loss(prediction, train_y[indices])
            else:
                loss = F.smooth_l1_loss(prediction, train_y[indices], beta=beta)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        model.eval()
        with torch.inference_mode():
            validation_loss = F.l1_loss(model(valid_x), valid_y).item()
        if validation_loss < best_loss - 1e-4:
            best_loss = validation_loss
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if patience >= 7:
            break
    model.load_state_dict(best_state)
    torch.save(best_state, f"refiner_{variant}_fold_{fold}.pth")
    print(f"Refiner {variant}, fold {fold}: validation MAE={best_loss:.4f} frames")
    return model


def prepare_refiners(image_files, images, probabilities, ground_truth):
    configs = (
        ("smooth", 42, 1e-3, 100, "smooth", 0.5),
        ("l1", 777, 8e-4, 900, "l1", 0.0),
        ("robust", 2026, 1.2e-3, 2000, "smooth", 1.0),
    )
    all_paths = [[f"refiner_{variant}_fold_{fold}.pth"
                  for fold in range(1, REFINER_FOLDS + 1)]
                 for variant, *_ in configs]
    if all(os.path.exists(path) for paths in all_paths for path in paths):
        model_groups = []
        for paths in all_paths:
            models = []
            for path in paths:
                model = BoundaryRefiner().to(device)
                model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
                model.eval()
                models.append(model)
            model_groups.append(models)
        print("Loaded three five-fold boundary-refiner ensembles")
        return model_groups

    features, targets, groups = build_refiner_samples(image_files, images, probabilities, ground_truth)
    print(f"Boundary-refiner samples: {len(targets)}")
    splits = list(GroupKFold(REFINER_FOLDS).split(features, targets, groups))
    model_groups = []
    for variant, model_seed, learning_rate, batch_seed, loss_kind, beta in configs:
        torch.manual_seed(model_seed)
        np.random.seed(model_seed)
        random.seed(model_seed)
        models = []
        for fold, (train_indices, valid_indices) in enumerate(splits, 1):
            models.append(train_refiner_fold(
                features[train_indices], targets[train_indices],
                features[valid_indices], targets[valid_indices], fold, variant,
                learning_rate, batch_seed, loss_kind, beta,
            ))
        model_groups.append(models)
    return model_groups


def refine_bounds(image, probability, bounds, refiner_groups):
    if not bounds:
        return []
    features = []
    for start, end in bounds:
        features.append(boundary_feature(image, probability, start, "start"))
        features.append(boundary_feature(image, probability, end, "end"))
    tensor = torch.from_numpy(np.stack(features)).to(device)
    with torch.inference_mode():
        offsets = sum(
            weight * torch.stack([model(tensor) for model in models]).mean(0)
            for weight, models in zip(REFINER_ENSEMBLE_WEIGHTS, refiner_groups)
        ).cpu().numpy()
    offsets = np.clip(offsets, -5, 5)
    offsets[0::2] *= START_OFFSET_SCALE
    offsets[1::2] *= END_OFFSET_SCALE
    duration = image.shape[1] * DT
    refined = []
    for index, (start, end) in enumerate(bounds):
        refined_start = np.clip((start + offsets[2 * index]) * DT, 0, duration)
        refined_end = np.clip((end + offsets[2 * index + 1]) * DT, 0, duration)
        if refined_end > refined_start:
            refined.append((refined_start, refined_end))
    return refined

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش مدل
</font>
</h2>



In [3]:
train_images, train_targets, ground_truth, train_groups = load_training_data(train_frame)
print(f"Loaded {len(train_images)} spectrograms in {len(train_groups)} width buckets")

models = prepare_base_models(train_images, train_targets, train_groups)
train_probabilities = ensemble_probabilities(models, train_images, train_groups)
refiners = prepare_refiners(
    train_frame.image_file.to_numpy(), train_images, train_probabilities, ground_truth
)
print("Base ensemble and three five-fold boundary refiners are ready.")


Loaded 2100 spectrograms in 11 width buckets
drone_model_1.pth epoch 01/12: loss=0.80705, seconds=16.0
drone_model_1.pth epoch 02/12: loss=0.38220, seconds=13.3
drone_model_1.pth epoch 03/12: loss=0.25342, seconds=13.4
drone_model_1.pth epoch 04/12: loss=0.20439, seconds=13.4
drone_model_1.pth epoch 05/12: loss=0.17663, seconds=13.8
drone_model_1.pth epoch 06/12: loss=0.11786, seconds=12.8
drone_model_1.pth epoch 07/12: loss=0.10550, seconds=13.1
drone_model_1.pth epoch 08/12: loss=0.07036, seconds=13.5
drone_model_1.pth epoch 09/12: loss=0.05755, seconds=13.6
drone_model_1.pth epoch 10/12: loss=0.05601, seconds=13.5
drone_model_1.pth epoch 11/12: loss=0.06305, seconds=13.5
drone_model_1.pth epoch 12/12: loss=0.05357, seconds=13.8
drone_model_2.pth epoch 01/12: loss=0.79818, seconds=16.9
drone_model_2.pth epoch 02/12: loss=0.40095, seconds=16.3
drone_model_2.pth epoch 03/12: loss=0.25435, seconds=16.3
drone_model_2.pth epoch 04/12: loss=0.18737, seconds=16.5
drone_model_2.pth epoch 05/

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>


In [4]:
results = []
test_files = sorted(name for name in os.listdir(TEST_DIR) if name.lower().endswith(".png"))
for filename in test_files:
    image_u8 = cv2.imread(os.path.join(TEST_DIR, filename), cv2.IMREAD_GRAYSCALE)
    if image_u8 is None:
        raise FileNotFoundError(filename)
    image = image_u8.astype(np.float32) / 255.0
    tensor = torch.from_numpy(image).unsqueeze(0).to(device)
    with torch.inference_mode():
        probability = sum(
            weight * torch.sigmoid(model(tensor)).squeeze(0)
            for weight, model in zip(ENSEMBLE_WEIGHTS, models)
        ).cpu().numpy()
    intervals = refine_bounds(image, probability, mask_bounds(probability), refiners)
    interval_text = ";".join(f"{start:.2f}-{end:.2f}" for start, end in intervals)
    results.append({
        "image_file": filename,
        "drone_count": len(intervals),
        "intervals": interval_text,
    })

submission = pd.DataFrame(results, columns=("image_file", "drone_count", "intervals"))
submission.to_csv("submission.csv", index=False)
print(submission.head(10))
print("Predicted drone-count distribution:")
print(submission.drone_count.value_counts().sort_index())
print(f"Generated submission.csv with {len(submission)} rows")


            image_file  drone_count                          intervals
0  sample_00420aba.png            2              8.04-8.98;13.99-15.04
1  sample_00d02e0f.png            2              5.47-6.44;10.14-11.83
2  sample_0192a5c5.png            2             8.96-10.54;15.93-16.97
3  sample_024162dc.png            0                                   
4  sample_029bebd2.png            1                          7.70-8.82
5  sample_029cef33.png            1                        13.81-14.96
6  sample_036b5c8a.png            2              2.68-3.51;14.26-15.88
7  sample_03a26644.png            2                2.13-4.44;8.86-9.66
8  sample_042d59d8.png            2              7.00-7.80;12.77-13.94
9  sample_044f7718.png            3  3.91-4.75;11.79-12.74;16.88-17.70
Predicted drone-count distribution:
drone_count
0     69
1    241
2    207
3    116
4     67
Name: count, dtype: int64
Generated submission.csv with 700 rows


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) در غیر این صورت، در پایان مسابقه نمره شما به صفر تغییر خواهد کرد.
    <br>
    همچنین اگر از کولب برای اجرای این فایل نوت‌بوک استفاده می‌کنید، قبل از ارسال فایل <code>result.zip</code>، آخرین نسخه‌ی نوت‌بوک خود را دانلود کرده و داخل فایل ارسالی قرار دهید.
</font>

In [5]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'drone.ipynb')):
    %notebook -e drone.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['drone.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['drone.ipynb', 'submission.csv']
